In [3]:
import pandas as pd
import numpy as np
import os

In [ ]:
metrics_df_fp = "../figs/c2ms_sim_exp/test_df.csv"
sims_latex_fp = "../figs/c2ms_sim_exp/sims_df.tex"
os_latex_fp = "../figs/c2ms_sim_exp/os_df.tex"
metrics_df = pd.read_csv(metrics_df_fp)
display(metrics_df)

In [5]:
split_type_to_name = {
    "inchikey": "InChIKey",
    "scaffold": "Scaffold",
}

model_type_to_name = {
    "fraggnn_d3": "FraGNNet-D3",
    "fraggnn_d4": "FraGNNet-D4",
    "neims": "NEIMS",
    "massformer": "MassFormer",
    "iceberg_inten3_100": "ICEBERG",
    "iceberg_inten3_all": "ICEBERG (+OptFrag)",
    "graff3": "GrAFF",
    "precursor": "Precursor-Only",
}

# add long ones
for k in list(model_type_to_name.keys()):
    model_type_to_name[f"{k}_long"] = model_type_to_name[k]

sims_latex_df = metrics_df[["split_type","model_type","cos_sim_0.01_mean","cos_hun_mean","cos_sim_0.01_std","cos_hun_std"]].copy()

#  latex_df = latex_df.drop(columns=["pred_oos_prob_mean","pred_oos_prob_std","pearson_r_oos_mean","pearson_r_oos_std"])
for metric in ["cos_sim_0.01","cos_hun"]:
    def fmt_func(row):
        mean_score = row[f"{metric}_mean"]
        std_score = row[f"{metric}_std"]
        if np.isnan(mean_score) or np.isnan(std_score):
            fmt_str = "-"
        else:
            fmt_str = f"${mean_score:.3f} \pm {std_score:.3f}$"
        return fmt_str
    sims_latex_df[metric] = sims_latex_df.apply(fmt_func, axis=1) 
    sims_latex_df = sims_latex_df.drop(columns=[f"{metric}_mean",f"{metric}_std"])
sims_latex_df.loc[:,"split_type"] = sims_latex_df["split_type"].replace(split_type_to_name)
sims_latex_df.loc[:,"model_type"] = sims_latex_df["model_type"].replace(model_type_to_name)
sims_latex_df.loc[~(sims_latex_df["model_type"].isin(["FraGNNet-D4","FraGNNet-D3","GrAFF","Precursor-Only"])),"cos_hun"] = "-"
# rename columns
sims_latex_df.columns = ["Split","Model",r"$\mathbb{C}_\text{BIN} \Uparrow$",r"$\mathbb{C}_\text{HUN} \Uparrow$"]
# latex_table = latex_df.to_latex(index=False, escape=False)

In [ ]:
display(sims_latex_df)

In [ ]:
# Pivot the DataFrame so that each Model appears once with separate columns for each Split.
# This creates a MultiIndex on the columns.
sims_latex_df = sims_latex_df.pivot(index='Model', columns='Split')

# Swap levels so that the 'Split' becomes the top level (desired for multirow format)
sims_latex_df = sims_latex_df.swaplevel(axis=1)

# Explicitly reorder the inner columns for each split: first the 0.01 column, then the HUN column.
sims_latex_df = sims_latex_df[[("InChIKey", r"$\mathbb{C}_\text{BIN} \Uparrow$"),
                     ("InChIKey", r"$\mathbb{C}_\text{HUN} \Uparrow$"),
                     ("Scaffold", r"$\mathbb{C}_\text{BIN} \Uparrow$"),
                     ("Scaffold", r"$\mathbb{C}_\text{HUN} \Uparrow$")]]


# Optionally, reset the index to have Model as a regular column
sims_latex_df.reset_index(inplace=True)

display(sims_latex_df)

Split               Model                         InChIKey  \
                          $\mathbb{C}_\text{BIN} \Uparrow$   
0             FraGNNet-D3                $0.721 \pm 0.002$   
1             FraGNNet-D4                $0.736 \pm 0.002$   
2                   GrAFF                $0.596 \pm 0.005$   
3                 ICEBERG                $0.707 \pm 0.001$   
4      ICEBERG (+OptFrag)                $0.732 \pm 0.000$   
5              MassFormer                $0.639 \pm 0.002$   
6                   NEIMS                $0.635 \pm 0.001$   
7          Precursor-Only                $0.319 \pm 0.000$   

Split                                                          Scaffold  \
      $\mathbb{C}_\text{HUN} \Uparrow$ $\mathbb{C}_\text{BIN} \Uparrow$   
0                    $0.693 \pm 0.002$                $0.659 \pm 0.001$   
1                    $0.709 \pm 0.002$                $0.678 \pm 0.005$   
2                    $0.602 \pm 0.004$                $0.520 \pm 0.005$   
3                                    -                $0.636 \pm 0.002$   
4                                    -                $0.668 \pm 0.002$   
5                                    -                $0.562 \pm 0.003$   
6                                    -                $0.546 \pm 0.003$   
7                    $0.285 \pm 0.000$                $0.313 \pm 0.000$   

Split                                   
      $\mathbb{C}_\text{HUN} \Uparrow$  
0                    $0.633 \pm 0.001$  
1                    $0.654 \pm 0.006$  
2                    $0.518 \pm 0.004$  
3                                    -  
4                                    -  
5                                    -  
6                                    -  
7                    $0.280 \pm 0.000$

In [8]:
# Export the pivoted DataFrame to LaTeX.
# The parameters `multicolumn=True` and `multirow=True` help format the header accordingly.
sims_latex_table = sims_latex_df.to_latex(index=False, multicolumn=True, multirow=True, escape=False)
with open(sims_latex_fp, "w") as f:
    f.write(sims_latex_table)

In [9]:
split_type_to_name = {
    "inchikey": "InChIKey",
    "scaffold": "Scaffold",
}

model_type_to_name = {
    "fraggnn_d3": "FraGNNet-D3",
    "fraggnn_d4": "FraGNNet-D4",
}

# add long ones
for k in list(model_type_to_name.keys()):
    model_type_to_name[f"{k}_long"] = model_type_to_name[k]

os_cols = ["split_type","model_type","true_oos_prob_mean","true_oos_prob_std","pred_oos_prob_mean","pred_oos_prob_std","tv_oos_mean","tv_oos_std"]
os_latex_df = metrics_df[os_cols].copy()

for metric in ["true_oos_prob","pred_oos_prob","tv_oos"]:
    def fmt_func(row):
        mean_score = row[f"{metric}_mean"]
        std_score = row[f"{metric}_std"]
        if np.isnan(mean_score) or np.isnan(std_score):
            fmt_str = "-"
        else:
            fmt_str = f"${mean_score:.3f} \pm {std_score:.3f}$"
        return fmt_str
    os_latex_df[metric] = os_latex_df.apply(fmt_func, axis=1) 
    os_latex_df = os_latex_df.drop(columns=[f"{metric}_mean",f"{metric}_std"])
os_latex_df.loc[:,"split_type"] = os_latex_df["split_type"].replace(split_type_to_name)
os_latex_df.loc[:,"model_type"] = os_latex_df["model_type"].replace(model_type_to_name)
os_latex_df = os_latex_df[os_latex_df["model_type"].isin(["FraGNNet-D4","FraGNNet-D3"])]
# rename columns
os_latex_df.columns = ["Split","Model",r"$P(M^{\text{OS}})$",r"$P_{\theta}(M^{\text{OS}})$",r"$\delta_{TV} \downarrow$"]

In [ ]:
display(os_latex_df)

,Split,Model,$P(M^{\text{OS}})$,$P_{\theta}(M^{\text{OS}})$,$\delta_{TV} \downarrow$
0,InChIKey,FraGNNet-D3,$0.171 \pm 0.000$,$0.191 \pm 0.007$,$0.080 \pm 0.002$
1,InChIKey,FraGNNet-D4,$0.098 \pm 0.000$,$0.115 \pm 0.003$,$0.057 \pm 0.001$
8,Scaffold,FraGNNet-D3,$0.193 \pm 0.000$,$0.213 \pm 0.008$,$0.094 \pm 0.002$
9,Scaffold,FraGNNet-D4,$0.110 \pm 0.000$,$0.127 \pm 0.002$,$0.067 \pm 0.001$


In [11]:
# Export the pivoted DataFrame to LaTeX.
# The parameters `multicolumn=True` and `multirow=True` help format the header accordingly.
os_latex_table = os_latex_df.to_latex(index=False, multicolumn=True, multirow=True, escape=False)
with open(os_latex_fp, "w") as f:
    f.write(os_latex_table)